# Ejercicio 7. La reconstrucción como proyección ortogonal

Trabajamos con la matriz A7, de 119 filas x 64 columnas, y con las mediciones con ruido de sigma = 0,06 del ejercicio anterior. La reconstrucción por mínimos cuadrados, que llamamos x_hat, minimiza ||A7 x - b_sigma||^2. El vector A7 x_hat es el punto del espacio columna de A7 que está más cerca de b_sigma.

### Código de los ejercicios anteriores

Para que el notebook sea autocontenido se repiten aca las piezas de los ejercicios 1 a 6 que se reutilizan: el fantoma, la función mascara_rayo, la función matriz_proyeccion, la matriz A7 y la generación de mediciones ruidosas.

Se fija una semilla (default_rng(912)) y se genera un único vector gaussiano z, de modo que el ruido de nivel sigma, que es sigma * escala * z, sea siempre el mismo ruido, sólo que más o menos intenso.

In [ ]:
import numpy as np

# fantoma y matrices de los ej anteriores, las redefino pq cada notebook tiene que andar solo
X_fantoma = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
    [0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
    [0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)
n = 8
x_fantoma = X_fantoma.reshape(-1)


def mascara_rayo(n, direccion, s):
    """Matriz n x n con 1 donde alpha*i + beta*j == s (centros de pixel)."""
    alpha, beta = direccion
    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    return (alpha * i + beta * j == s).astype(float)


def matriz_proyeccion(n, direcciones):
    """Devuelve (A, rayos): A en R^{m x n^2} y la lista [((alpha, beta), s), ...]."""
    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    filas, rayos = [], []

    for (alpha, beta) in direcciones:
        valores_s = np.unique(alpha * i + beta * j)  # ordenados de menor a mayor
        for s in valores_s:
            filas.append(mascara_rayo(n, (alpha, beta), s).reshape(-1))
            rayos.append(((alpha, beta), int(s)))
    return np.array(filas), rayos


D2 = [(1, 0), (0, 1)]
D4 = D2 + [(1, 1), (1, -1)]
D7 = D4 + [(2, 1), (1, 2), (3, 1)]
A7, rayos7 = matriz_proyeccion(n, D7)

# Mediciones ideales y ruido (misma semilla y mismo z en todos los ejercicios)
b7 = A7 @ x_fantoma #en el ej3 multiplicamos con matmul, pero quise probar usar @ aca
escala = np.sqrt(np.mean(b7 ** 2))
# misma semilla que en el ej 6, si no los numeros no coinciden con los de alla
rng = np.random.default_rng(912)
z = rng.standard_normal(b7.shape)


def medicion_ruidosa(sigma):
    return b7 + sigma * escala * z

## Inciso a) 

In [ ]:
sigma = 0.06
b_sigma = medicion_ruidosa(sigma)

x_hat = np.linalg.lstsq(A7, b_sigma, rcond=None)[0] #el coso para hacer min cuadrados
# el residuo es lo que le sobra a b_sigma despues de ajustar
r_sigma = b_sigma - A7 @ x_hat 

print("m, n^2:", A7.shape)
print("b_sigma:", np.linalg.norm(b_sigma))
print("r_sigma:", np.linalg.norm(r_sigma))
print("A7^T r_sigma:", np.linalg.norm(A7.T @ r_sigma))

m, n^2: (119, 64)
b_sigma: 11.954129458948286
r_sigma: 0.5335971499194508
A7^T r_sigma: 4.146134045189627e-14


Resultado. ||A7^T r_sigma|| es del orden de 1e-14, es decir, diminuta. Cada componente de A7^T r_sigma es el producto interno entre una columna de A7 y el residuo, y todas son numéricamente nulas.

## Inciso b) 

Ortogonalidad a todas las columnas. La componente j de A7^T r_sigma es el producto interno entre la columna j de A7 y r_sigma. Que A7^T r_sigma = 0 significa entonces que r_sigma es perpendicular a cada columna de A7, para todo j de 0 a 63. Como el espacio columna de A7 es el generado por esas 64 columnas, un vector perpendicular a todos los generadores es perpendicular a toda combinación lineal de ellos. Es decir, r_sigma pertenece al complemento ortogonal del espacio columna de A7, que es igual al espacio nulo de A7^T.

Es la proyección. Escribimos b_sigma = A7 x_hat + r_sigma, donde A7 x_hat está en el espacio columna de A7 y r_sigma está en su complemento ortogonal. Esa descomposición existe y es única (teorema de la proyección ortogonal), y la componente que está en el subespacio es la proyección ortogonal de b_sigma sobre el espacio columna de A7. Por lo tanto, A7 x_hat es esa proyección.

Además, para cualquier otro x = x_hat + delta vale, por Pitágoras, que ||b_sigma - A7 x||^2 = ||r_sigma||^2 + ||A7 delta||^2 >= ||r_sigma||^2. Esto es así porque A7 delta está en el espacio columna de A7 y r_sigma es perpendicular a ese espacio. Entonces A7 x_hat es el punto más cercano a b_sigma.

Las ecuaciones A7^T (b_sigma - A7 x_hat) = 0 son justamente las ecuaciones normales: A7^T A7 x_hat = A7^T b_sigma. El gradiente de (1/2) ||A7 x - b_sigma||^2 se anula exactamente cuando el residuo es ortogonal al espacio columna de A7.

Verificamos esto numéricamente:

In [ ]:
# Matriz de proyección ortogonal sobre Col(A7): P = A (A^T A)^{-1} A^T  (A7 tiene rango columna completo)
P = A7 @ np.linalg.solve(A7.T @ A7, A7.T)

# P la armo solo para chequear, en la practica no hace falta
print("||P b - A x_hat||            :", np.linalg.norm(P @ b_sigma - A7 @ x_hat))

||P b - A x_hat||            : 1.281964053379236e-14


## Inciso c) 

Son dos usos distintos de la palabra proyección.

La proyección tomográfica es la medición física: la suma de la atenuación a lo largo de un rayo, es decir, b_r = suma sobre todos los píxeles j de a_rj * x_j. Un conjunto de rayos paralelos con el mismo ángulo theta forma una proyección. Se describe con la matriz A, que lleva la imagen a las mediciones. Es el modelo directo, y no es una matriz cuadrada ni idempotente. Su rol es generar los datos, b = A x, y su entrada es la imagen desconocida x.

La proyección ortogonal de mínimos cuadrados es una operación geométrica en el espacio de las mediciones: encuentra el punto del espacio columna de A que está más cerca de un vector dado. Se describe con la matriz P = A (A^T A)^(-1) A^T, que va del espacio de las mediciones a sí mismo y cumple que P^2 = P y que P^T = P. Su rol aparece al resolver el problema inverso, donde se plantea A x_hat = P b. Su entrada son las mediciones ya obtenidas, b_sigma, que tienen ruido.

La conexión: el escáner proyecta, en sentido tomográfico, la imagen para producir b. Como b_sigma tiene ruido y puede no estar en el espacio columna de A7, mínimos cuadrados proyecta ortogonalmente b_sigma sobre el conjunto de mediciones que realmente podría producir alguna imagen, que es el espacio columna de A7. Recién entonces se resuelve A7 x_hat = P b_sigma, que siempre tiene solución y es única porque A7 tiene rango columna completo.